In [0]:
%sql
USE CATALOG gold_dev;
USE SCHEMA analytics;


In [0]:
%sql
CREATE OR REPLACE VIEW vw_sales_base AS
SELECT
    f.order_id,

    -- Dates
    od.date AS order_date,
    sd.date AS ship_date,
    DATEDIFF(sd.date, od.date) AS days_to_ship,

    -- Customer
    c.customer_key,
    c.customer_id,
    c.customer_name,
    c.customer_segment,
    c.region,

    -- Product
    p.product_key,
    p.product_id,
    p.product_name,
    p.category,
    p.sub_category,

    -- Measures
    f.order_quantity,
    f.sales_amount,
    f.discount_amount,
    f.profit_amount,

    -- Metadata
    f.ingestion_ts,
    f.load_timestamp

FROM silver_dev.global_mart_retail.fact_sales f

JOIN silver_dev.global_mart_retail.dim_customer c
  ON f.customer_key = c.customer_key
 AND c.is_current_record = true

JOIN silver_dev.global_mart_retail.dim_product p
  ON f.product_key = p.product_key
 AND p.is_current_record = true

JOIN silver_dev.global_mart_retail.dim_date od
  ON f.order_date_key = od.date_key

JOIN silver_dev.global_mart_retail.dim_date sd
  ON f.ship_date_key = sd.date_key;


In [0]:
%sql
CREATE OR REPLACE VIEW vw_profitability AS
SELECT
    YEAR(order_date)  AS year,
    MONTH(order_date) AS month,

    category,
    sub_category,

    -- Quantities & Sales
    SUM(order_quantity)          AS total_order_quantity,
    SUM(sales_amount) AS total_sales,

    -- Profit
    SUM(profit_amount) AS total_profit,

    -- Margins
    ROUND(
        SUM(profit_amount) / NULLIF(SUM(sales_amount), 0),
        4
    ) AS profit_margin,

    ROUND(
        SUM(profit_amount) / NULLIF(SUM(sales_amount), 0) * 100,
        2
    ) AS profit_margin_pct,

    -- Returns
    SUM(
        CASE
            WHEN sales_amount < 0 THEN ABS(sales_amount)
            ELSE 0
        END
    ) AS total_return,

    ROUND(
        COUNT(CASE WHEN sales_amount < 0 THEN 1 END)
        / NULLIF(COUNT(*), 0),
        4
    ) AS return_rate

FROM gold_dev.analytics.vw_sales_base
GROUP BY
    YEAR(order_date),
    MONTH(order_date),
    category,
    sub_category;


In [0]:
%sql
CREATE OR REPLACE VIEW vw_logistics_efficiency AS
SELECT
    YEAR(order_date)  AS year,
    MONTH(order_date) AS month,

    region,

    ROUND(AVG(days_to_ship), 2) AS avg_days_to_ship,
    MAX(days_to_ship)           AS max_days_to_ship,
    MIN(days_to_ship)           AS min_days_to_ship

FROM vw_sales_base
GROUP BY
    YEAR(order_date),
    MONTH(order_date),
    region;


In [0]:
%sql
CREATE OR REPLACE VIEW vw_customer_value AS
SELECT
    d.year,
    d.month,

    c.customer_key,
    c.customer_name,
    c.region,

    SUM(f.sales_amount)  AS total_sales,
    SUM(f.profit_amount) AS total_profit,
    COUNT(DISTINCT f.order_id) AS total_unique_orders,
    COUNT(*) AS total_orders_entries,
    SUM(f.order_quantity)      AS total_quantity,

    -- Average Order Value
    ROUND(
        SUM(f.sales_amount) / NULLIF(COUNT(DISTINCT f.order_id), 0),
        2
    ) AS avg_order_value,

    -- Ranking per Year
    ROW_NUMBER() OVER (
        PARTITION BY d.year
        ORDER BY SUM(f.sales_amount) DESC
    ) AS rn

FROM silver_dev.global_mart_retail.fact_sales f

JOIN silver_dev.global_mart_retail.dim_customer c
  ON f.customer_key = c.customer_key
 AND c.is_current_record = true

JOIN silver_dev.global_mart_retail.dim_date d
  ON f.order_date_key = d.date_key

GROUP BY
    d.year,
    d.month,
    c.customer_key,
    c.customer_name,
    c.region;
